In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)  # Load environment variables from .env file

In [ ]:
openai_api_key= os.getenv("OPENAI_API_KEY")
anthropic_api_key= os.getenv("ANTHROPIC_API_KEY")
gemini_api_key= os.getenv("GEMINI_API_KEY")
deepseek_api_key= os.getenv("DEEPSEEK_API_KEY")
groq_api_key= os.getenv("GROQ_API_KEY")
grok_api_key= os.getenv("GROK_API_KEY")
openrouter_api_key= os.getenv("OPENROUTER_API_KEY") 

if openai_api_key:
    print("OpenAI API Key is set.")
else:
    print("OpenAI API Key is NOT set.")
    
if anthropic_api_key:
    print("Anthropic API Key is set.")
else:
    print("Anthropic API Key is NOT set.")

if gemini_api_key:
    print("Gemini API Key is set.")
else:
    print("Gemini API Key is NOT set.")

if deepseek_api_key:
    print("DeepSeek API Key is set.")
else:
    print("DeepSeek API Key is NOT set.")

if groq_api_key:
    print("GROQ API Key is set.")
else:
    print("GROQ API Key is NOT set.")

if grok_api_key:
    print("GROK API Key is set.")
else:
    print("GROK API Key is NOT set.")

if openrouter_api_key:
    print("OpenRouter API Key is set.")
else:
    print("OpenRouter API Key is NOT set.")


In [ ]:
request="""
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request+="Answer only with the question, no explanation."
message=[{"role":"user","content":request}]


In [ ]:
openai = OpenAI()

In [ ]:
response =openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=message
)

question= response.choices[0].message.content
display(Markdown(question))

In [21]:
anthropic =OpenAI(api_key=anthropic_api_key, base_url=os.getenv("ANTHROPIC_BASE_URL"))
gemini =OpenAI(api_key=gemini_api_key, base_url=os.getenv("GEMINI_BASE_URL"))
deepseek =OpenAI(api_key=deepseek_api_key, base_url=os.getenv("DEEPSEEK_BASE_URL"))
groq =OpenAI(api_key=groq_api_key, base_url=os.getenv("GROQ_BASE_URL"))
grok =OpenAI(api_key=grok_api_key, base_url=os.getenv("GROK_BASE_URL"))
openrouter =OpenAI(api_key=openrouter_api_key, base_url=os.getenv("OPENROUTER_BASE_URL"))
#ollama =OpenAI(api_key=ollama, base_url=os.getenv("OLLAMA_BASE_URL"))



In [ ]:
competitors=[]
answers=[]
message = [{"role":"user","content":question}]


In [22]:
def record(competitor, answer):
    competitors.append(competitor)
    answers.append(answer)
    display(Markdown(f"**{competitor}**: {answer}"))
    

In [ ]:
model_name="gpt-4o-mini"

response=openai.chat.completions.create(
    model=model_name,
    messages=message)
answer=response.choices[0].message.content

record(model_name, answer)



In [ ]:
model_name="claude-3.0"
response=anthropic.chat.completions.create(
    model=model_name,
    messages=message)
answer=response.choices[0].message.content

record(model_name, answer)

In [ ]:
model_name="gemini-1.5-turbo"
response=gemini.chat.completions.create(
    model=model_name,  
    messages=message)
answer=response.choices[0].message.content

record(model_name, answer)


In [ ]:
model_name="deepseek-1.0"
response = deepseek.chat.completions.create(
    model=model_name,
    messages=message    
)

answer= response.choices[0].message.content

record(model_name, answer)


In [ ]:
model_name="grok-1.0"
response= grok.chat.completions.create(
    model=model_name,
    messages=message
)

answer = response.choices[0].message.content
record(model_name, answer)

In [ ]:
def get_response(model_name, message):
    if model_name.startswith("gpt"):
        response=openai.chat.completions.create(
            model=model_name,
            messages=message)
        answer=response.choices[0].message.content
    elif model_name.startswith("claude"):
        response=anthropic.chat.completions.create(
            model=model_name,
            messages=message)
        answer=response.choices[0].message.content
    elif model_name.startswith("gemini"):
        response=gemini.chat.completions.create(
            model=model_name,  
            messages=message)
        answer=response.choices[0].message.content
    elif model_name.startswith("deepseek"):
        response = deepseek.chat.completions.create(
            model=model_name,
            messages=message    
        )
        answer= response.choices[0].message.content
    elif model_name.startswith("grok"):
        response= grok.chat.completions.create(
            model=model_name,
            messages=message
        )
        answer = response.choices[0].message.content
    else:
        answer="Model not recognized."
    
    record(model_name, answer)

In [ ]:
print(len(competitors))
print(competitors)
print(answers)


In [ ]:
for competitor , answer in zip(competitors, answers):
    print(f"{competitor}: {answer}")
    

In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

print(together)

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
judge_message = [{"role":"user","content":judge}]

In [ ]:

model_name = "grok-4.3"

response = grok.chat.completions.create(
    model=model_name,
    messages=judge_message
)

result= response.choices[0].message.content
print(result)